# Anchor Assignments

Method Overview:
1. Preprocessing each image using green channel + CLAHE (Contrast Limited Adaptive Histogram Equalization) + resizing
2. Extract local features using SIFT
3. Match descriptors with Lowe's ratio test
4. Fit a homography with RANSAC
5. Score each anchor-test pair using:
   - number of inliers
   - inlier ratio
   - reprojection error
   - vessel-strucutre (Frangi) similarity
6. Assign each test image to the anchor with the best score



## Kernel Check

In [14]:
import sys
print(sys.executable)

import cv2
print(cv2.__version__)

/opt/miniconda3/envs/mia/bin/python
4.13.0


## Imports & Requirements

In [15]:
#!/usr/bin/env python3
"""
Requirements to install:
    pip install opencv-python numpy pandas
"""

from __future__ import annotations

import argparse
import os
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import cv2
import numpy as np
import pandas as pd


VALID_EXTENSIONS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}

## Argument Parsing & Handling Image Files

In [16]:
# Parse inputs to be given in main method
def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Assign each test fundus image to an anchor image.")
    parser.add_argument("--anchors_dir", type=str, required=True, help="Directory containing anchor images")
    parser.add_argument("--tests_dir", type=str, required=True, help="Directory containing test images")
    parser.add_argument("--output_csv", type=str, required=True, help="Path to save grouping.csv")
    parser.add_argument("--diagnostics_csv", type=str, default=None, help="Optional path to save detailed pairwise scores")
    parser.add_argument("--feature", type=str, default="sift", choices=["sift"], help="Feature extractor to use") # Choices if we want to experiment different feature detectors
    parser.add_argument("--resize", type=int, default=512, help="Resize images to resize x resize before matching")
    parser.add_argument("--ratio_thresh", type=float, default=0.75, help="Lowe ratio test threshold")
    parser.add_argument("--ransac_thresh", type=float, default=5.0, help="RANSAC reprojection threshold in pixels")
    parser.add_argument("--clahe_clip", type=float, default=2.0, help="CLAHE clip limit")
    parser.add_argument("--nfeatures", type=int, default=2000, help="Number of features for SIFT")
    return parser.parse_args()


# Confirm valid image file path
def is_image_file(path: Path) -> bool:
    return path.is_file() and path.suffix.lower() in VALID_EXTENSIONS


# Sort image files
def list_image_files(folder: Path) -> List[Path]:
    files = [p for p in sorted(folder.iterdir()) if is_image_file(p)]
    if not files:
        raise FileNotFoundError(f"No image files found in: {folder}")
    return files


# Read image from disk with OpenCV
def load_image(path: Path) -> np.ndarray:
    img = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if img is None:
        raise ValueError(f"Failed to read image: {path}")
    return img

## Pre-processing, Feature Detecting, Matching Feature Descriptors

In [17]:
# Suppress dark border/background
# Return binary retinal mask
def create_fundus_mask(gray: np.ndarray) -> np.ndarray:
    mask = (gray > 20).astype(np.uint8) * 255 # Thresholding
    kernel = np.ones((7, 7), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel) # Morphological Closing
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)  # Morphological Opening
    return mask


# Preprocessing
# Return preprocessed grayscale image (proc_img) and binary mask of relevant (non-background) fundus region (mask)
def preprocess_fundus(img_bgr: np.ndarray, resize_to: int = 512, clahe_clip: float = 2.0) -> Tuple[np.ndarray, np.ndarray]:
    if img_bgr.ndim != 3 or img_bgr.shape[2] != 3:
        raise ValueError("Expected a color BGR image")

    # OpenCV loads BGR; green channel is index 1
    green = img_bgr[:, :, 1]

    # Resize first for consistency
    proc = cv2.resize(green, (resize_to, resize_to), interpolation=cv2.INTER_AREA)

    # CLAHE contrast enhancement
    clahe = cv2.createCLAHE(clipLimit=clahe_clip, tileGridSize=(8, 8))
    proc = clahe.apply(proc)

    # Create mask from enhanced image
    mask = create_fundus_mask(proc)

    return proc, mask


# Vessel enhancement using Sobel + CLAHE
def enhance_vessels(img):
    img = img.astype(np.float32) / 255.0                # Convert image to float32 & normalize
    sobelx = cv2.Sobel(img, cv2.CV_32F, 1, 0, ksize=3)  # Horizontal gradient (highlight vertical edges)
    sobely = cv2.Sobel(img, cv2.CV_32F, 0, 1, ksize=3)  # Vertical gradient (highlight horizontal edges)
    vessel = np.sqrt(sobelx**2 + sobely**2)             # Gradient magnitude
    return vessel


### Perhaps sub out above method for true frangi enhancement ###


# Create SIFT feature detector
def make_detector(feature_type: str, nfeatures: int):
    if feature_type == "sift":  # We can try different feature detectors later if wanted
        if not hasattr(cv2, "SIFT_create"): # Safety check to ensure correct cv2 build
            raise RuntimeError("SIFT is not available in your OpenCV build --> Install a build that includes SIFT")
        return cv2.SIFT_create(nfeatures=nfeatures)
    raise ValueError(f"Unsupported feature type: {feature_type}")


# Return key points and computed descriptors around each keypoint given the image and mask
def extract_features(img: np.ndarray, mask: np.ndarray, detector) -> Tuple[List[cv2.KeyPoint], Optional[np.ndarray]]:
    kp, des = detector.detectAndCompute(img, mask)
    return kp, des



# Match feature descriptors between 2 images, returns list of reliable feature matches
def match_descriptors(des1: Optional[np.ndarray], des2: Optional[np.ndarray], feature_type: str, ratio_thresh: float) -> List[cv2.DMatch]:
    if des1 is None or des2 is None:  # i.e. No matches if no descriptors
        return []                   

    if len(des1) < 2 or len(des2) < 2: # Too few matches, need two nearest neighbors
        return []

    if feature_type == "sift":
        # FLANN (Fast Library for Approximate Nearest Neighbors) mactcher for float descriptors
        index_params = dict(algorithm=1, trees=5)  # KD-tree --> K dimensional tree, splitting data along different dimensions to compare points, eliminating searching through regions with bad matches
        search_params = dict(checks=50)
        matcher = cv2.FlannBasedMatcher(index_params, search_params)
        knn_matches = matcher.knnMatch(des1, des2, k=2) # Find two nearest neighbors

    else:
        raise ValueError(f"Unsupported feature type: {feature_type}") # Safety check

    good_matches: List[cv2.DMatch] = [] # Initialize list of good matches
    for pair in knn_matches:
        if len(pair) < 2:
            continue
        m, n = pair # m --> best match, n --> second best match
        if m.distance < ratio_thresh * n.distance:  # Lowe's ratio test, keeping matches where best match is significantly better than second best
            good_matches.append(m)

    return good_matches

## asdf

In [18]:
# Compute reprojection error of how given transformation H aligns point sets 1 and 2
def compute_reprojection_error(pts1: np.ndarray, pts2: np.ndarray, H: np.ndarray) -> float:
    if len(pts1) == 0:  # If no points, there is an infinite reprojection error (i.e. meaningless alignment)
        return float("inf")

    pts1_proj = cv2.perspectiveTransform(pts1.reshape(-1, 1, 2), H).reshape(-1, 2)  # Apply transform H to pts1
    pts2 = pts2.reshape(-1, 2) # Reshape for comparison
    err = np.sqrt(np.sum((pts1_proj - pts2) ** 2, axis=1)) # Euclidean distance between transformed points (pts1_proj) and actual corresponding points (pts2)
    return float(np.mean(err)) if len(err) > 0 else float("inf") # Return avg error across all points


# Take in a list of matched key points, estimate homographies between anchor and test images with RANSAC, return inliers, inlier ratio, and avg reprojection error
def ransac_score(kp1: List[cv2.KeyPoint], kp2: List[cv2.KeyPoint], matches: List[cv2.DMatch], ransac_thresh: float) -> Dict[str, float]:
    # kp1/matches query points from anchor image
    # kp2/matches train points from test image
    
    # Initialize a dictionary to store total matches, number of inliers, inlier ratio, avg reprojection error
    stats = { "num_matches": float(len(matches)),  "num_inliers": 0.0, "inlier_ratio": 0.0, "mean_error": float("inf")}

    # Need at least four matches to compute homography
    if len(matches) < 4:
        return stats

    # Extract matched point coordinates
    pts1 = np.float32([kp1[m.queryIdx].pt for m in matches]).reshape(-1, 1, 2)
    pts2 = np.float32([kp2[m.trainIdx].pt for m in matches]).reshape(-1, 1, 2)

    # Use RANSAC to estimate homography H and identify inliers (i.e. consistent matches)
    H, mask = cv2.findHomography(pts1, pts2, cv2.RANSAC, ransac_thresh)

    # If homography fails return default stats
    if H is None or mask is None:
        return stats

    inlier_mask = mask.ravel().astype(bool)             # Convert RANSAC to boolean mask
    num_inliers = int(np.sum(inlier_mask))              # Count number of inliers
    inlier_ratio = num_inliers / max(len(matches), 1)   # Compute fraction of consistent matches

    # Compute average reprojection error for inliers only
    if num_inliers > 0:
        pts1_in = pts1[inlier_mask]
        pts2_in = pts2[inlier_mask]
        mean_error = compute_reprojection_error(pts1_in, pts2_in, H)
    else:
        mean_error = float("inf")  # Infinite error if no inliers

    # Update dictionary
    stats["num_inliers"] = float(num_inliers)
    stats["inlier_ratio"] = float(inlier_ratio)
    stats["mean_error"] = float(mean_error)

    return stats # All metrics


# Similarity score of two images' vessel structures
def vessel_similarity(img1, img2):
    v1 = enhance_vessels(img1).ravel() # Apply vessel enhancement and flatten to 1D vector for comparison
    v2 = enhance_vessels(img2).ravel()
    if np.std(v1) < 1e-6 or np.std(v2) < 1e-6:  # If almost no variation in image (i.e. constant image) there should be no similarity
        return 0.0

    return float(np.corrcoef(v1, v2)[0, 1])  # Pearson correlation coefficient (1 for very similar, 0 for no relationship, -1 for opposite patterns)


# Weigh metrics to compute final similarity score (weights have been & can be further optimized)
def final_pair_score(stats: Dict[str, float], vessel_sim= float) -> float:
    mean_error = stats["mean_error"] # Avg reprojection error
    if np.isinf(mean_error) or np.isnan(mean_error):
        mean_error = 1e6    # Huge value to penalize bad matches if invalid error

    # Combined score
    score = (
        3.0 * stats["num_inliers"] # Reward more inliers
        + 40.0 * stats["inlier_ratio"] # Reward high fraction of consistent matches
        - 8.0 * mean_error # Penalize poor geometric alignment
        + 40.0 * vessel_sim  # Reward vessel similarity
    )

    return float(score)


# Cache to locally & temporarily store feature data (path, pre-processed image, mask, keypoints, descriptors)
def build_feature_cache(image_paths: List[Path], detector, resize_to: int, clahe_clip: float) -> Dict[str, Dict[str, object]]:
    cache: Dict[str, Dict[str, object]] = {}

    for path in image_paths:
        img_bgr = load_image(path)
        proc, mask = preprocess_fundus(img_bgr, resize_to=resize_to, clahe_clip=clahe_clip)
        kp, des = extract_features(proc, mask, detector)

        cache[path.name] = {"path": path, "proc": proc, "mask": mask, "kp": kp, "des": des,}

        print(f"Cached {path.name}: {len(kp)} keypoints")

    return cache


# Assign a test image to its best anchor and return statistics & diagnostics
def assign_single_test_image(test_name: str, test_entry: Dict[str, object], anchor_cache: Dict[str, Dict[str, object]], feature_type: str, ratio_thresh: float, ransac_thresh: float) -> Tuple[str, Dict[str, float], List[Dict[str, object]]]:
    pairwise_rows: List[Dict[str, object]] = []    # List for anchor-test comparison
    best_anchor_name: Optional[str] = None         # Initialize anchor name
    best_score = -float("inf")                     # Initialize best score (start low)
    best_stats: Optional[Dict[str, float]] = None  # Initialize stats for best match

    # Extract key points, descriptors, and processed image for test image
    kp_test = test_entry["kp"]
    des_test = test_entry["des"]
    test_img = test_entry["proc"]

    # Look through all anchor images
    for anchor_name, anchor_entry in anchor_cache.items():
        # Extract key points, descriptors, and processed image for anchor image
        kp_anchor = anchor_entry["kp"]
        des_anchor = anchor_entry["des"]
        anchor_img = anchor_entry["proc"]

        # Match features between anchor and test
        matches = match_descriptors(des_anchor, des_test, feature_type=feature_type, ratio_thresh=ratio_thresh)

        # Use RANSAC to find num inliers, inlier ratio, reprojection error
        stats = ransac_score(kp_anchor, kp_test, matches, ransac_thresh=ransac_thresh)

        vessel_sim = vessel_similarity(anchor_img, test_img) # Gloabl vessel similarity
        score = final_pair_score(stats, vessel_sim)          # Combine metrics into final score

        # Dictionary storing test image, anchor image, score, and all metrics
        row = {
            "test_image": test_name,
            "anchor_image": anchor_name,
            "score": score,
            "num_matches": int(stats["num_matches"]),
            "num_inliers": int(stats["num_inliers"]),
            "inlier_ratio": float(stats["inlier_ratio"]),
            "mean_error": float(stats["mean_error"]),
            "vessel_sim": float(vessel_sim),
        }
        pairwise_rows.append(row) # Add results to list

        # Compare to existing best anchor image and re-assign if better match (i.e. if higher score)
        if score > best_score:
            best_score = score
            best_anchor_name = anchor_name
            best_stats = stats

    if best_anchor_name is None or best_stats is None: # Failure case if no valid assignment
        raise RuntimeError(f"Failed to assign anchor for test image: {test_name}")

    return best_anchor_name, best_stats, pairwise_rows

## Assigning Images to Anchors!

In [19]:
def main() -> None:
    # Change paths for different devices / when we get testing data
    anchors_dir = Path("/Users/nataliesmith/Desktop/MIA Project 1/example test/anchor_images")
    tests_dir = Path("/Users/nataliesmith/Desktop/MIA Project 1/example test/test_images")
    output_csv = Path("/Users/nataliesmith/Desktop/MIA Project 1/example test/grouping_results.csv")
    diagnostics_csv = Path("/Users/nataliesmith/Desktop/MIA Project 1/example test/diagnostics_results.csv")

    # Params chosen to maximize performance
    feature = "sift"
    resize = 768
    ratio_thresh = 0.70
    ransac_thresh = 3.0
    clahe_clip = 2.0
    nfeatures = 4000

    # Safety check
    if not anchors_dir.exists():
        raise FileNotFoundError(f"Anchors directory does not exist: {anchors_dir}")
    if not tests_dir.exists():
        raise FileNotFoundError(f"Tests directory does not exist: {tests_dir}")

    anchor_paths = list_image_files(anchors_dir)
    test_paths = list_image_files(tests_dir)

    # Confirm correct number of anchor and test images
    print(f"Found {len(anchor_paths)} anchor images")
    print(f"Found {len(test_paths)} test images")

    # Feature detector w chosen parameters
    detector = make_detector(feature, nfeatures)

    # Chache's for anchor and test images
    print("\nBuilding anchor cache...")
    anchor_cache = build_feature_cache(anchor_paths, detector=detector, resize_to=resize, clahe_clip=clahe_clip)

    print("\nBuilding test cache...")
    test_cache = build_feature_cache(test_paths, detector=detector, resize_to=resize, clahe_clip=clahe_clip)

    grouping_rows = []
    diagnostics_rows = []

    # Anchor assignment
    print("\nAssigning anchors...")
    for test_name, test_entry in test_cache.items():
        assigned_anchor, stats, pairwise_rows = assign_single_test_image(test_name=test_name, test_entry=test_entry, anchor_cache=anchor_cache, feature_type=feature, ratio_thresh=ratio_thresh, ransac_thresh=ransac_thresh)

        print(
            f"{test_name} -> {assigned_anchor} | "
            f"inliers={int(stats['num_inliers'])}, "
            f"inlier_ratio={stats['inlier_ratio']:.3f}, "
            f"mean_error={stats['mean_error']:.3f}"
        )

        grouping_rows.append({"test_image": test_name,"anchor_image": assigned_anchor,})
        diagnostics_rows.extend(pairwise_rows)


    output_csv.parent.mkdir(parents=True, exist_ok=True)  # Ensure output directory exists
    grouping_df = pd.DataFrame(grouping_rows)             # Convert results to pandas DataFrame
    grouping_df.to_csv(output_csv, index=False)           # Save as .csv
    print(f"\nSaved grouping CSV to: {output_csv}")       # Print location of csv

    diagnostics_csv.parent.mkdir(parents=True, exist_ok=True)
    diagnostics_df = pd.DataFrame(diagnostics_rows)
    diagnostics_df.sort_values(["test_image", "score"], ascending=[True, False], inplace=True)  # Sort by test_image and score (highest first)
    diagnostics_df.to_csv(diagnostics_csv, index=False)
    print(f"Saved diagnostics CSV to: {diagnostics_csv}")


if __name__ == "__main__":
    main()

Found 5 anchor images
Found 25 test images

Building anchor cache...
Cached anchor_01.tiff: 2122 keypoints
Cached anchor_02.tiff: 664 keypoints
Cached anchor_03.tiff: 1229 keypoints
Cached anchor_04.tiff: 1112 keypoints
Cached anchor_05.tiff: 1084 keypoints

Building test cache...
Cached test_01.tiff: 206 keypoints
Cached test_02.tiff: 678 keypoints
Cached test_03.tiff: 3604 keypoints
Cached test_04.tiff: 1977 keypoints
Cached test_05.tiff: 756 keypoints
Cached test_06.tiff: 656 keypoints
Cached test_07.tiff: 1084 keypoints
Cached test_08.tiff: 3992 keypoints
Cached test_09.tiff: 680 keypoints
Cached test_10.tiff: 1706 keypoints
Cached test_11.tiff: 1245 keypoints
Cached test_12.tiff: 2217 keypoints
Cached test_13.tiff: 2017 keypoints
Cached test_14.tiff: 1225 keypoints
Cached test_15.tiff: 1157 keypoints
Cached test_16.tiff: 1111 keypoints
Cached test_17.tiff: 538 keypoints
Cached test_18.tiff: 3985 keypoints
Cached test_19.tiff: 1129 keypoints
Cached test_20.tiff: 575 keypoints
Cache